# 03 — Il confronto a parità di genere (Versione 3)

Tutti i modelli finiti, per testata, sulle 304 testate valutate. Metrica principale (fissata prima): **media sugli
orientamenti** dell'accuratezza bilanciata; poi l'AUC media sugli orientamenti, il dettaglio per orientamento e le
metriche su tutto. Per ModernBERT con più seed c'è l'**ensemble** (media delle probabilità, deciso prima).

Intervalli al 95% con un **bootstrap per gruppo** (2000 ricampionamenti) e differenze con `feature_genere_tuned`
(se c'è, altrimenti `feature_genere`) sugli stessi ricampionamenti. Gira da solo dopo ogni esperimento della coda.

In [ ]:
import json
import re

import numpy as np
import pandas as pd

from comune import RESULTS, VERSIONI, metriche, per_testata

modelli = {}
for d in sorted(RESULTS.iterdir()):
    if d.is_dir() and d.name != "transformer" and (d / "FATTO").exists():
        modelli[d.name] = pd.read_csv(d / "articoli.csv")
for d in sorted((RESULTS / "transformer").glob("*")):
    if (d / "FATTO").exists():
        modelli[d.name] = pd.read_csv(d / "articoli.csv")
seed = {}
for n in list(modelli):
    m = re.match(r"(.+)_s\d+$", n)
    if m:
        seed.setdefault(m[1], []).append(n)
for config, nomi in seed.items():
    if len(nomi) > 1:
        parti = [modelli[n].set_index("article_id") for n in nomi]
        e = parti[0][["domain", "fold", "y", "orientamento"]].copy(); e["p"] = sum(p.loc[e.index, "p"] for p in parti) / len(parti)
        modelli[f"{config}_ensemble{len(nomi)}"] = e.reset_index()
# la combinazione fissata la sera del 20 ottobre (dopo aver visto il seed 42, prima degli altri seed): media 0,5/0,5
# delle probabilità del TF-IDF di riferimento e dell'ensemble di ModernBERT-base (tutti i seed finiti)
ens = [n for n in modelli if n.startswith("modernbert_base_genere_ensemble")]
if "feature_genere_tuned" in modelli and ens:
    a, b = modelli["feature_genere_tuned"].set_index("article_id"), modelli[ens[0]].set_index("article_id")
    e = a[["domain", "fold", "y", "orientamento"]].copy(); e["p"] = 0.5 * a.p + 0.5 * b.loc[a.index, "p"]
    modelli["combinazione_tfidf_modernbert"] = e.reset_index()
print("modelli finiti:", list(modelli))
T = {n: per_testata(p) for n, p in modelli.items()}
tutto = pd.DataFrame({n: metriche(t) for n, t in T.items()}).T
tutto.to_csv(RESULTS / "03_genere_metriche.csv")
tutto.round(3)

In [ ]:
# per orientamento, quante testate e il dettaglio di ogni modello
print(T[next(iter(T))].groupby("orientamento").y.agg(testate="size", affidabili="sum"))
tutto[[c for c in tutto.columns if c.startswith("bilanciata")]].round(2)

In [ ]:
B = 2000
rif = "feature_genere_tuned" if "feature_genere_tuned" in T else "feature_genere"
gruppo = pd.read_csv(VERSIONI / "fold_finale.csv").set_index("domain").gruppo
dom = T[rif].index; g = gruppo.reindex(dom).to_numpy(); gruppi = np.unique(g)
pos = {x: np.where(g == x)[0] for x in gruppi}
rng = np.random.default_rng(0)
campioni = [np.concatenate([pos[x] for x in rng.choice(gruppi, len(gruppi))]) for _ in range(B)]


def principale(t, idx):
    return metriche(t.iloc[idx])["media sugli orientamenti"]


boot = {n: np.array([principale(t.reindex(dom), idx) for idx in campioni]) for n, t in T.items()}
righe = {}
for n in T:
    b = boot[n]
    r = {"media sugli orientamenti": tutto.loc[n, "media sugli orientamenti"], "2.5%": np.nanpercentile(b, 2.5), "97.5%": np.nanpercentile(b, 97.5)}
    if n != rif:
        d = b - boot[rif]
        r.update({"Δ vs TF-IDF": tutto.loc[n, "media sugli orientamenti"] - tutto.loc[rif, "media sugli orientamenti"],
                  "Δ 2.5%": np.nanpercentile(d, 2.5), "Δ 97.5%": np.nanpercentile(d, 97.5), "P(Δ>0)": float(np.nanmean(d > 0))})
    righe[n] = r
iv = pd.DataFrame(righe).T
iv.to_csv(RESULTS / "03_genere_intervalli.csv")
print("riferimento:", rif)
iv.round(3)